[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/31_gradient_accumulation.ipynb)

# 🟢 简单: 梯度累积

实现一个**带有梯度累积的训练步骤**——在内存有限的情况下模拟大批次训练。

### 函数签名
```python
def accumulated_step(model, optimizer, 损失_fn, micro_batches) -> float:
    # micro_batches: list of (input, target) tuples
    # 返回: 平均损失 (float)
```

### 算法
1. `optimizer.zero_grad()`
2. 对于 micro_batches 中的每个 `(x, y)`: `损失 = 损失_fn(model(x), y) / len(micro_batches)`, 然后 `损失.backward()`
3. `optimizer.step()`
4. 返回总累积损失

关键思路：在 backward 前将每个损失除以 `n` 使累积梯度等于单个大批次的梯度。

In [ ]:
# 在 Colab 中安装 torch-judge（在 JupyterLab/Docker 中无操作）
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass

In [ ]:
import torch
import torch.nn as nn

In [ ]:
# ✏️ 在此实现你的代码

def accumulated_step(model, optimizer, 损失_fn, micro_batches):
    pass  # zero_grad, 循环 (前向, 缩放损失, 反向), step

原理: 将一个大批次改成多个小批次计算，减少中间参数存储数量，将每个小批次损失缩放小批次数量。

假设我们有一个总批次包含 $N$ 个样本，分为 $n$ 个微批次，每个微批次有 $m = N/n$ 个样本。

### 标准大批次训练

1. 损失函数：$L_{\text{total}} = \frac{1}{N} \sum_{i=1}^{N} \ell(y_i, \hat{y}_i)$

2. 梯度：$\nabla L_{\text{total}} = \frac{1}{N} \sum_{i=1}^{N} \nabla \ell(y_i, \hat{y}_i)$

3. 参数更新：$\theta_{\text{new}} = \theta - \eta \cdot \nabla L_{\text{total}}$

### 梯度累积训练

1. 每个微批次的损失（缩放后）：$L_{\text{micro}_j} = \frac{1}{n} \cdot L_{\text{total}_j}$,  其中 $L_{\text{total}_j}$ 是第 $j$ 个微批次未缩放的损失：$L_{\text{total}_j} = \frac{1}{m} \sum_{i=1}^{m} \ell(y_i, \hat{y}_i)$,  因此：$L_{\text{micro}_j} = \frac{1}{n} \cdot \frac{1}{m} \sum_{i=1}^{m} \ell(y_i, \hat{y}_i) = \frac{1}{N} \sum_{i=1}^{m} \ell(y_i, \hat{y}_i)$

2. 每个微批次的梯度：$\nabla L_{\text{micro}_j} = \frac{1}{N} \sum_{i=1}^{m} \nabla \ell(y_i, \hat{y}_i)$, 累积 $n$ 个微批次的梯度：$\sum_{j=1}^{n} \nabla L_{\text{micro}_j} = \sum_{j=1}^{n} \frac{1}{N} \sum_{i=1}^{m} \nabla \ell(y_i, \hat{y}_i)$

3. 由于所有微批次覆盖了全部 $N$ 个样本：$= \frac{1}{N} \sum_{i=1}^{N} \nabla \ell(y_i, \hat{y}_i) = \nabla L_{\text{total}} $


梯度累积的核心就是**用时间换空间**：通过多次前向-反向传播来模拟大批次训练，同时保持数学上的等价性。

In [ ]:
def standard_step(model, optimizer, loss_fn, batch):
    """
    标准大批次训练
    
    Args:
        model: PyTorch模型
        optimizer: 优化器
        loss_fn: 损失函数
        batch: (x, y) 包含所有数据
    """
    # 清零梯度
    optimizer.zero_grad()
    
    # 前向传播
    output = model(batch[0])
    
    # 计算损失（不除以任何东西）
    loss = loss_fn(output, batch[1]) # 总损失 / n
    
    # 反向传播
    loss.backward()
    
    # 更新参数
    optimizer.step()
    
    return loss.item()

def accumulated_step(model, optimizer, loss_fn, micro_batches) -> float:
    """
    执行一个带有梯度累积的训练步骤
    
    Args:
        model: PyTorch模型
        optimizer: 优化器
        loss_fn: 损失函数
        micro_batches: 由 (输入, 目标) 元组组成的列表，每个是一个小批次
    
    Returns:
        float: 所有微批次上的平均损失值
    """
    # 1. 清零梯度，避免上一次迭代的梯度累积
    optimizer.zero_grad()
    
    total_loss = 0.0
    # n * m 为总样本数
    n = len(micro_batches) # 小批次数量
    m = micro_batches[0][0].shape[0] # 每个小批次的样本数量
    
    # 2. 遍历每个微批次
    for x, y in micro_batches:

        output = model(x)
        
        # 计算损失，并除以小批次数量以模拟大批次梯度
        loss = loss_fn(output, y) / n # 总损失 / m / n
        
        # 反向传播，累积梯度
        loss.backward()
        
        # 累加损失值（用于返回）
        total_loss += loss.item() * n  # 恢复原始损失值
    
    # 3. 将每个小批次的梯度计算完成再更新模型参数
    optimizer.step()
    
    # 4. 返回平均损失
    return total_loss / n

In [ ]:
# 🧪 调试
model = nn.Linear(4, 2)
opt = torch.optim.SGD(model.parameters(), lr=0.01)
损失 = accumulated_step(model, opt, nn.MSELoss(),
    [(torch.randn(2, 4), torch.randn(2, 2)) for _ in range(4)])
print('损失:', 损失)

In [ ]:
# ✅ 提交
from torch_judge import check
check('gradient_accumulation')